# Unit Testing with `unittest`

A **unit test** checks a small, isolated piece of code, usually a single function or class method. Each test should follow the same rhythm: arrange the inputs, act by calling the code, and assert the expected result.

Python's standard library includes `unittest`, so no additional dependency is required.

In this notebook, we will cover:
1. Defining a test case and running it.
2. Using common assertion methods.
3. Sharing setup with `setUp()`.
4. Testing expected exceptions.

## 1. Defining a Test Case

A test case is a class that inherits from `unittest.TestCase`. Every method whose name begins with `test_` is discovered and run as a separate test.

In [1]:
def calculate_accuracy(correct_predictions, total_predictions):
    if total_predictions == 0:
        raise ValueError("total_predictions must be greater than zero")
    return correct_predictions / total_predictions

Each test method focuses on one behavior. Keeping tests small makes it obvious which expectation failed when a test does not pass.

In [2]:
import unittest


class TestCalculateAccuracy(unittest.TestCase):
    def test_returns_expected_fraction(self):
        self.assertAlmostEqual(calculate_accuracy(8, 10), 0.8)

    def test_zero_correct_predictions(self):
        self.assertEqual(calculate_accuracy(0, 10), 0.0)

    def test_perfect_accuracy(self):
        self.assertEqual(calculate_accuracy(10, 10), 1.0)

## 2. Running the Tests

`unittest` can discover tests from a `TestCase` class. In a script, `unittest.main()` is common; inside a notebook, a `TextTestRunner` gives us control over the output.

In [3]:
import io
import re


def run_tests(test_case):
    suite = unittest.TestLoader().loadTestsFromTestCase(test_case)
    output = io.StringIO()
    result = unittest.TextTestRunner(stream=output, verbosity=2).run(suite)
    clean_output = re.sub(r"\x1b\[[0-9;]*m", "", output.getvalue())
    print(f"Tests run: {result.testsRun}")
    print(clean_output)


run_tests(TestCalculateAccuracy)

Tests run: 3
test_perfect_accuracy (__main__.TestCalculateAccuracy.test_perfect_accuracy) ... ok
test_returns_expected_fraction (__main__.TestCalculateAccuracy.test_returns_expected_fraction) ... ok
test_zero_correct_predictions (__main__.TestCalculateAccuracy.test_zero_correct_predictions) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.000s

OK



## 3. Sharing Setup with `setUp()`

When several tests use the same input, `setUp()` runs before every test method. This keeps fixtures consistent while ensuring that each test starts with a fresh state.

In [4]:
class TestAccuracyWithSetup(unittest.TestCase):
    def setUp(self):
        self.correct = 45
        self.total = 50

    def test_accuracy_is_ninety_percent(self):
        self.assertAlmostEqual(calculate_accuracy(self.correct, self.total), 0.9)

    def test_accuracy_is_between_zero_and_one(self):
        accuracy = calculate_accuracy(self.correct, self.total)
        self.assertGreaterEqual(accuracy, 0.0)
        self.assertLessEqual(accuracy, 1.0)


run_tests(TestAccuracyWithSetup)

Tests run: 2
test_accuracy_is_between_zero_and_one (__main__.TestAccuracyWithSetup.test_accuracy_is_between_zero_and_one) ... ok
test_accuracy_is_ninety_percent (__main__.TestAccuracyWithSetup.test_accuracy_is_ninety_percent) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.000s

OK



## 4. Testing Expected Exceptions

Errors are part of a function's contract. Use `assertRaises()` to confirm that invalid input produces the expected exception instead of allowing the test to fail unexpectedly.

In [5]:
class TestAccuracyValidation(unittest.TestCase):
    def test_rejects_zero_total_predictions(self):
        with self.assertRaises(ValueError):
            calculate_accuracy(3, 0)

    def test_error_message_is_clear(self):
        with self.assertRaisesRegex(ValueError, "greater than zero"):
            calculate_accuracy(3, 0)


run_tests(TestAccuracyValidation)

Tests run: 2
test_error_message_is_clear (__main__.TestAccuracyValidation.test_error_message_is_clear) ... ok
test_rejects_zero_total_predictions (__main__.TestAccuracyValidation.test_rejects_zero_total_predictions) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.000s

OK

